In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Removing Duplicates**

In [0]:
df = spark.sql("SELECT * FROM `ete-catalog`.silver.customers")

In [0]:
df.limit(10).display()

# **Filtering Old and New Records**

**Using Table for filering**

In [0]:
flag = int(dbutils.widgets.get('flag'))

if flag == 0 :
    df_old = spark.sql(''' SELECT DimCustomerKey, customer_id, create_date, update_date
                       FROM `ete-catalog`.gold.DimCustomers
                       ''')
else :
    df_old = spark.sql(''' SELECT 0 as DimCustomerKey, 0 as customer_id, 0 as create_date, 0 as update_date
                       FROM `ete-catalog`.silver.region
                       WHERE 1=0
                       ''')
    
    df_old = df_old.withColumn('create_date', to_timestamp(col('create_date')))


In [0]:
df_old.display()

**Renaming These old columns with old as preceeding**

In [0]:
df_old = df_old.withColumnsRenamed({"DimCustomerKey" : "old_DimCustomerKey",
                           "customer_id" : "old_customer_id",
                           'create_date' : 'old_create_date',
                           "update_date" : "old_update_date"})

**Joining both dfs**

In [0]:
df_join = df.join(df_old,df['customer_id'] == df_old['old_customer_id'], 'left')

display(df_join.limit(12))

**Seperating old and new records**

In [0]:
df_new = df_join.filter(col('old_DimCustomerKey').isNull())

df_old = df_join.filter(col('old_DimCustomerKey').isNotNull())

In [0]:
df_new.display()

**Preparing df_old**

In [0]:
# Droping unwanted columns or redundent columns
df_old = df_old.drop('old_customer_id','old_update_date')

#Renaming the columns 

df_old = df_old.withColumnsRenamed({
    "old_DimCustomerKey" : "DimCustomerKey",
    "old_create_date" : "create_date"
})

# Creating update_date column with current date

df_old = df_old.withColumn('update_date', current_timestamp())


**Preaping df_new**

In [0]:
#Droping unwanted columns
df_new = df_new.drop('old_DimCustomerKey','old_customer_id','old_create_date','old_update_date')

#Recreating date columns 

df_new = df_new.withColumns({
    'create_date' : current_timestamp(),
    'update_date' : current_timestamp()
})


In [0]:
# Creating DimCustomerKey starting from df_old max_val in df_new

if flag == 1 :
    max_val = 0
else :
    df_max_val = spark.sql("SELECT max(DimCustomerKey) as max_key from `ete-catalog`.gold.DimCustomers")

    max_val = df_max_val.collect()[0]['max_key']

df_new = df_new.withColumn('DimCustomerKey',monotonically_increasing_id()+lit(max_val)+lit(1))

In [0]:
df_new.limit(3).display()